In [2]:
import sys
from pathlib import Path
import pandas as pd
from contextlib import contextmanager

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

@contextmanager
def full_df_display():
    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.expand_frame_repr", False,
    ):
        yield

%load_ext autoreload
%autoreload 2

path: /workspace


In [3]:
from src.data_download.companies_house import build_download_manifest
company_numbers = ["08948140", "11270200"] # company_name: Finance Advice Centre LTD
FilingList = build_download_manifest(company_numbers)
print(len(FilingList))

13


In [4]:
from src.data_download.companies_house import download_ixbrl_filing
for filing in FilingList:
    download_ixbrl_filing(filing)

2025-10-06, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzQ4Mzg0NDYxN2FkaXF6a2N4.ixbrl
2025-08-26, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzQ3ODU4MjE4N2FkaXF6a2N4.ixbrl
2025-08-26, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzQ3ODU4MjE4N2FkaXF6a2N4.ixbrl
2024-11-27, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzQ0NTExOTQ1NmFkaXF6a2N4.ixbrl
2024-11-27, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzQ0NTExOTQ1NmFkaXF6a2N4.ixbrl
2024-07-01, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzQyNzE3ODU4MGFkaXF6a2N4.ixbrl
2024-07-01, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzQyNzE3ODU4MGFkaXF6a2N4.ixbrl
2023-10-18, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzM5NzAzMzA5NmFkaXF6a2N4.ixbrl
2023-10-18, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzM5NzAzMzA5NmFkaXF6a2N4.ixbrl
2

In [5]:
from src.parsing.ixbrl_loader import (load_ixbrl, extract_contexts, 
                                      extract_facts, extract_units)
import pandas as pd

ixbrl_folders = projectRoot / "data" / "raw" / "companies_house"
df_contexts = []
df_facts = []
df_units = []

for company_dir in ixbrl_folders.iterdir():
    if company_dir.is_dir() or company_dir.name != "debug":
        for ixbrl_filepath in company_dir.glob("*.ixbrl"):
            ixbrl_doc= load_ixbrl(ixbrl_filepath)
            df_contexts.append((ixbrl_filepath.stem,
                              extract_contexts(ixbrl_doc)))
            df_facts.append((ixbrl_filepath.stem,
                              extract_facts(ixbrl_doc)))
            df_units.append((ixbrl_filepath.stem,
                              extract_units(ixbrl_doc)))

In [6]:
filing_id, ctx_df = df_contexts[1]
display(ctx_df.head(5))
# with full_df_display():
    # print(ctx_df.loc[ctx_df["context_id"] == "CURRENT_FY_END"])


filing_id, facts_df = df_facts[1]
display(facts_df.head(5))
# with full_df_display():
    # print(facts_df.loc[facts_df["contextRef"] == "CURRENT_FY_END"].head(5))

filing_id, units_df = df_units[1]
display(units_df.head(5))
facts = facts_df.merge(units_df, left_on="unitRef", right_on="unit_id", how="left")
display(facts.head(5))

,context_id,entity_identifier,entity_scheme,start_date,end_date,instant,scenario
0,CURRENT_FY_START,08948140,http://www.companieshouse.gov.uk/,None,None,2018-03-31,None
1,CURRENT_FY_END,08948140,http://www.companieshouse.gov.uk/,None,None,2019-03-31,None
2,CURRENT_FY_PERIOD,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None
3,CURRENT_FY_END_CURRENT,08948140,http://www.companieshouse.gov.uk/,None,None,2019-03-31,None
4,CURRENT_FY_PERIOD_CURRENT,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None


,name,contextRef,unitRef,decimals,value
0,frs-core:PropertyPlantEquipment,CURRENT_FY_END,GBP,0,"8,983"
1,frs-core:PropertyPlantEquipment,PREVIOUS_FY_END,GBP,0,"5,575"
2,frs-core:FixedAssets,CURRENT_FY_END,GBP,0,"8,983"
3,frs-core:FixedAssets,PREVIOUS_FY_END,GBP,0,"5,575"
4,frs-core:Debtors,CURRENT_FY_END,GBP,0,"152,456"


,unit_id,measure
0,GBP,iso4217:GBP
1,EUR,iso4217:EUR
2,USD,iso4217:USD
3,Shares,xbrli:shares
4,Pure,xbrli:pure


,name,contextRef,unitRef,decimals,value,unit_id,measure
0,frs-core:PropertyPlantEquipment,CURRENT_FY_END,GBP,0,"8,983",GBP,iso4217:GBP
1,frs-core:PropertyPlantEquipment,PREVIOUS_FY_END,GBP,0,"5,575",GBP,iso4217:GBP
2,frs-core:FixedAssets,CURRENT_FY_END,GBP,0,"8,983",GBP,iso4217:GBP
3,frs-core:FixedAssets,PREVIOUS_FY_END,GBP,0,"5,575",GBP,iso4217:GBP
4,frs-core:Debtors,CURRENT_FY_END,GBP,0,"152,456",GBP,iso4217:GBP


In [7]:
# Create a dictionary to store the joined tables for each filing
facts_with_context_dict = {}

for (filing_id, facts_df), (_, units_df), (_, ctx_df) in zip(df_facts, df_units, df_contexts):
    # Merge facts with units
    facts = facts_df.merge(units_df, left_on="unitRef", right_on="unit_id", how="left")
    # Merge with context
    facts_with_context = facts.merge(
        ctx_df,
        left_on="contextRef",
        right_on="context_id",
        how="left"
    )
    # Store the result
    facts_with_context_dict[filing_id] = facts_with_context

# Example: display the first few rows for each filing
for filing_id, df in facts_with_context_dict.items():
    print(f"Facts with context for {filing_id}")
    display(df.head(5))

Facts with context for MzE4NTE0NDIyM2FkaXF6a2N4


,name,contextRef,unitRef,decimals,value,unit_id,measure,context_id,entity_identifier,entity_scheme,start_date,end_date,instant,scenario
0,core:PropertyPlantEquipment,CurrYearEnd,GBP,0,"1,299",GBP,iso4217:GBP,CurrYearEnd,08948140,http://www.companieshouse.gov.uk/,None,None,2017-03-31,None
1,core:Debtors,CurrYearEnd,GBP,0,"145,146",GBP,iso4217:GBP,CurrYearEnd,08948140,http://www.companieshouse.gov.uk/,None,None,2017-03-31,None
2,core:Debtors,CurrYearStart,GBP,0,"109,326",GBP,iso4217:GBP,CurrYearStart,08948140,http://www.companieshouse.gov.uk/,None,None,2016-03-31,None
3,core:CashBankOnHand,CurrYearEnd,GBP,0,"210,787",GBP,iso4217:GBP,CurrYearEnd,08948140,http://www.companieshouse.gov.uk/,None,None,2017-03-31,None
4,core:CurrentAssets,CurrYearEnd,GBP,0,"355,933",GBP,iso4217:GBP,CurrYearEnd,08948140,http://www.companieshouse.gov.uk/,None,None,2017-03-31,None


Facts with context for MzI1MjkyNzY4N2FkaXF6a2N4


,name,contextRef,unitRef,decimals,value,unit_id,measure,context_id,entity_identifier,entity_scheme,start_date,end_date,instant,scenario
0,frs-core:PropertyPlantEquipment,CURRENT_FY_END,GBP,0,"8,983",GBP,iso4217:GBP,CURRENT_FY_END,08948140,http://www.companieshouse.gov.uk/,None,None,2019-03-31,None
1,frs-core:PropertyPlantEquipment,PREVIOUS_FY_END,GBP,0,"5,575",GBP,iso4217:GBP,PREVIOUS_FY_END,08948140,http://www.companieshouse.gov.uk/,None,None,2018-03-31,None
2,frs-core:FixedAssets,CURRENT_FY_END,GBP,0,"8,983",GBP,iso4217:GBP,CURRENT_FY_END,08948140,http://www.companieshouse.gov.uk/,None,None,2019-03-31,None
3,frs-core:FixedAssets,PREVIOUS_FY_END,GBP,0,"5,575",GBP,iso4217:GBP,PREVIOUS_FY_END,08948140,http://www.companieshouse.gov.uk/,None,None,2018-03-31,None
4,frs-core:Debtors,CURRENT_FY_END,GBP,0,"152,456",GBP,iso4217:GBP,CURRENT_FY_END,08948140,http://www.companieshouse.gov.uk/,None,None,2019-03-31,None


Facts with context for MzI4MTcyOTA3NWFkaXF6a2N4


...

,name,contextRef,unitRef,decimals,value,unit_id,measure,context_id,entity_identifier,entity_scheme,start_date,end_date,instant,scenario
0,core:PropertyPlantEquipment,CurrYearEnd,GBP,0,"2,092",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2023-03-31,None
1,core:PropertyPlantEquipment,CurrYearStart,GBP,0,"1,585",GBP,iso4217:GBP,CurrYearStart,11270200,http://www.companieshouse.gov.uk/,None,None,2022-03-31,None
2,core:Debtors,CurrYearEnd,GBP,0,"21,653",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2023-03-31,None
3,core:Debtors,CurrYearStart,GBP,0,"28,837",GBP,iso4217:GBP,CurrYearStart,11270200,http://www.companieshouse.gov.uk/,None,None,2022-03-31,None
4,core:CashBankOnHand,CurrYearEnd,GBP,0,"43,202",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2023-03-31,None


Facts with context for MzQ0NTExOTQ1NmFkaXF6a2N4


,name,contextRef,unitRef,decimals,value,unit_id,measure,context_id,entity_identifier,entity_scheme,start_date,end_date,instant,scenario
0,core:PropertyPlantEquipment,CurrYearEnd,GBP,0,"1,633",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2024-03-31,None
1,core:PropertyPlantEquipment,CurrYearStart,GBP,0,"2,092",GBP,iso4217:GBP,CurrYearStart,11270200,http://www.companieshouse.gov.uk/,None,None,2023-03-31,None
2,core:Debtors,CurrYearEnd,GBP,0,"15,223",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2024-03-31,None
3,core:Debtors,CurrYearStart,GBP,0,"21,653",GBP,iso4217:GBP,CurrYearStart,11270200,http://www.companieshouse.gov.uk/,None,None,2023-03-31,None
4,core:CashBankOnHand,CurrYearEnd,GBP,0,"43,864",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2024-03-31,None


Facts with context for MzQ4Mzg0NDYxN2FkaXF6a2N4


,name,contextRef,unitRef,decimals,value,unit_id,measure,context_id,entity_identifier,entity_scheme,start_date,end_date,instant,scenario
0,core:PropertyPlantEquipment,CurrYearEnd,GBP,0,"1,225",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2025-03-31,None
1,core:PropertyPlantEquipment,CurrYearStart,GBP,0,"1,633",GBP,iso4217:GBP,CurrYearStart,11270200,http://www.companieshouse.gov.uk/,None,None,2024-03-31,None
2,core:Debtors,CurrYearEnd,GBP,0,"17,519",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2025-03-31,None
3,core:Debtors,CurrYearStart,GBP,0,"15,223",GBP,iso4217:GBP,CurrYearStart,11270200,http://www.companieshouse.gov.uk/,None,None,2024-03-31,None
4,core:CashBankOnHand,CurrYearEnd,GBP,0,"24,843",GBP,iso4217:GBP,CurrYearEnd,11270200,http://www.companieshouse.gov.uk/,None,None,2025-03-31,None


In [ ]:


# Example: display the structure of the facts_with_context for each filing
for filing_id, facts_with_context in facts_with_context_dict.items():
    print(f"\nFiling ID: {filing_id}")
    print("Type:", type(facts_with_context))
    print("Shape:", facts_with_context.shape)
    print("Columns:", facts_with_context.columns.tolist())

import os

# Ensure the processed directory exists
processed_dir = projectRoot / "data" / "processed"
os.makedirs(processed_dir, exist_ok=True)

# Save each dataframe to a CSV file
for filing_id, df in facts_with_context_dict.items():
    csv_path = processed_dir / f"{filing_id}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")



Filing ID: MzE4NTE0NDIyM2FkaXF6a2N4
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (67, 14)
Columns: ['name', 'contextRef', 'unitRef', 'decimals', 'value', 'unit_id', 'measure', 'context_id', 'entity_identifier', 'entity_scheme', 'start_date', 'end_date', 'instant', 'scenario']

Filing ID: MzI1MjkyNzY4N2FkaXF6a2N4
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (111, 14)
Columns: ['name', 'contextRef', 'unitRef', 'decimals', 'value', 'unit_id', 'measure', 'context_id', 'entity_identifier', 'entity_scheme', 'start_date', 'end_date', 'instant', 'scenario']

Filing ID: MzI4MTcyOTA3NWFkaXF6a2N4
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (109, 14)
Columns: ['name', 'contextRef', 'unitRef', 'decimals', 'value', 'unit_id', 'measure', 'context_id', 'entity_identifier', 'entity_scheme', 'start_date', 'end_date', 'instant', 'scenario']

Filing ID: MzI5OTI3NjMxNGFkaXF6a2N4
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (109, 14)
Columns: ['name', 'contextRef', 'unitRef', 'deci

In [8]:
from src.parsing.facts_extractor import extract_canonical_facts, tag_mapping

df = facts_with_context_dict["MzI1MjkyNzY4N2FkaXF6a2N4"]
# display(df.head(5))

tag_map = tag_mapping()
# print(tag_map)

# print(df)
extractor = extract_canonical_facts(df, tag_map)


TESTING extract_canonical_facts function
DEBUG: Unique tag values in DataFrame: ['frs-core:PropertyPlantEquipment' 'frs-core:FixedAssets'
 'frs-core:Debtors' 'frs-core:CashBankOnHand' 'frs-core:CurrentAssets'
 'frs-core:Creditors' 'frs-core:NetCurrentAssetsLiabilities'
 'frs-core:TotalAssetsLessCurrentLiabilities'
 'frs-core:NetAssetsLiabilities' 'frs-core:Equity'
 'frs-core:PropertyPlantEquipmentGrossCost'
 'frs-core:TotalAdditionsIncludingFromBusinessCombinationsPropertyPlantEquipment'
 'frs-core:AccumulatedDepreciationImpairmentPropertyPlantEquipment'
 'frs-core:IncreaseFromDepreciationChargeForYearPropertyPlantEquipment'
 'frs-core:PrepaymentsAccruedIncome' 'frs-core:OtherDebtors'
 'frs-core:CorporationTaxPayable'
 'frs-core:OtherTaxationSocialSecurityPayable' 'frs-core:OtherCreditors'
 'frs-core:AccruedLiabilitiesDeferredIncome'
 'frs-core:AmountsOwedToDirectors' 'frs-bus:NameProductionSoftware'
 'frs-bus:VersionProductionSoftware' 'frs-bus:AccountingStandardsApplied'
 'frs-bus:Co

In [45]:
with full_df_display():
   print(extractor)

                                  metric    value currency period_end                   context_id                                  source_tag                                                                                                                                                                                                                                                                                                                                                                                            raw_row
0               property_plant_equipment    8,983      GBP       None               CURRENT_FY_END             frs-core:PropertyPlantEquipment               {'name': 'frs-core:PropertyPlantEquipment', 'contextRef': 'CURRENT_FY_END', 'unitRef': 'GBP', 'decimals': '0', 'value': '8,983', 'unit_id': 'GBP', 'measure': 'iso4217:GBP', 'context_id': 'CURRENT_FY_END', 'entity_identifier': '08948140', 'entity_scheme': 'http://www.companieshouse.gov.uk/', 'start_date': None, '